# Day 16: Intelligent Document Automation – Resume OCR System
Name: Divyashree R

## Project Overview

This project implements an **OCR-based Intelligent Document Processing (IDP) system** for extracting structured information from resumes.

## Information Extracted

The system extracts the following fields from resumes:

- Name
- Email
- Phone
- Skills
- Education
- Experience

In [7]:
# Install kagglehub
!pip install kagglehub

import kagglehub
import os

# Download dataset from Kaggle
dataset_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

print("Dataset downloaded to:", dataset_path)

# List dataset files
for root, dirs, files in os.walk(dataset_path):
    for file in files[:10]:
        print(os.path.join(root, file))

Using Colab cache for faster access to the 'resume-dataset' dataset.
Dataset downloaded to: /kaggle/input/resume-dataset
/kaggle/input/resume-dataset/Resume/Resume.csv
/kaggle/input/resume-dataset/data/data/DESIGNER/22506245.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/13998435.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/67582956.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/34349255.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/26790545.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/12674307.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/11807040.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/44145704.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/27497542.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/11155153.pdf
/kaggle/input/resume-dataset/data/data/BPO/95625660.pdf
/kaggle/input/resume-dataset/data/data/BPO/15573418.pdf
/kaggle/input/resume-dataset/data/data/BPO/57706851.pdf
/kaggle/input/resume-dataset/data/data/BPO/16492045.pd

In [8]:
!pip install pdf2image
!apt-get install poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (170 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [9]:
import os
import random

DATASET_PATH = dataset_path

resume_files = []

# Collect resume PDFs
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith(".pdf"):
            resume_files.append(os.path.join(root, file))

print("Total resumes found:", len(resume_files))

# Select only 15 resumes
selected_resumes = random.sample(resume_files, min(15, len(resume_files)))

print("\nSelected resumes:")
for r in selected_resumes:
    print(r)

Total resumes found: 2484

Selected resumes:
/kaggle/input/resume-dataset/data/data/ARTS/22182279.pdf
/kaggle/input/resume-dataset/data/data/FITNESS/52684666.pdf
/kaggle/input/resume-dataset/data/data/AUTOMOBILE/22732234.pdf
/kaggle/input/resume-dataset/data/data/DIGITAL-MEDIA/25038571.pdf
/kaggle/input/resume-dataset/data/data/FINANCE/24530382.pdf
/kaggle/input/resume-dataset/data/data/PUBLIC-RELATIONS/31211074.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/14014749.pdf
/kaggle/input/resume-dataset/data/data/APPAREL/26691587.pdf
/kaggle/input/resume-dataset/data/data/CONSTRUCTION/44147689.pdf
/kaggle/input/resume-dataset/data/data/CHEF/25953797.pdf
/kaggle/input/resume-dataset/data/data/DESIGNER/78149576.pdf
/kaggle/input/resume-dataset/data/data/DIGITAL-MEDIA/27080812.pdf
/kaggle/input/resume-dataset/data/data/AVIATION/29595906.pdf
/kaggle/input/resume-dataset/data/data/HR/28828844.pdf
/kaggle/input/resume-dataset/data/data/DIGITAL-MEDIA/28609364.pdf


In [10]:
from pdf2image import convert_from_path

IMAGE_DATASET = "/content/resume_images"
os.makedirs(IMAGE_DATASET, exist_ok=True)

image_paths = []

for i, pdf_path in enumerate(selected_resumes):

    images = convert_from_path(pdf_path)

    img_path = os.path.join(IMAGE_DATASET, f"resume_{i}.png")

    images[0].save(img_path, "PNG")

    image_paths.append(img_path)

print("Converted resumes to images:", len(image_paths))

Converted resumes to images: 15


In [11]:
import cv2

PROCESSED_PATH = "/content/processed_images"
os.makedirs(PROCESSED_PATH, exist_ok=True)

def preprocess_image(image_path):

    image = cv2.imread(image_path)

    # Step 1: Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Step 2: Noise reduction
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    # Step 3: Adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    return thresh


for img_path in image_paths:

    processed = preprocess_image(img_path)

    filename = os.path.basename(img_path)

    save_path = os.path.join(PROCESSED_PATH, filename)

    cv2.imwrite(save_path, processed)

print("Image preprocessing completed.")

Image preprocessing completed.


In [1]:
!apt-get install tesseract-ocr
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [12]:
import pytesseract
import os
from PIL import Image

OCR_OUTPUT = "/content/ocr_output"
os.makedirs(OCR_OUTPUT, exist_ok=True)

processed_images = os.listdir(PROCESSED_PATH)

print("Total processed images:", len(processed_images))


for img_file in processed_images:

    img_path = os.path.join(PROCESSED_PATH, img_file)

    img = Image.open(img_path)

    text = pytesseract.image_to_string(img)

    txt_file = img_file.replace(".png", ".txt")

    with open(os.path.join(OCR_OUTPUT, txt_file), "w") as f:
        f.write(text)

print("OCR extraction completed.")

Total processed images: 15
OCR extraction completed.


In [14]:
!pip install -q -U google-generativeai

In [16]:
import google.generativeai as genai
import json
import os
import time

genai.configure(api_key="GEMINI_API_KEY_WAS_PUT_HERE")

model = genai.GenerativeModel("gemini-2.5-flash")

JSON_OUTPUT = "/content/json_output"
os.makedirs(JSON_OUTPUT, exist_ok=True)


def extract_resume_information(text):

    prompt = f"""
    Extract the following details from the resume text.

    Fields:
    Name
    Email
    Phone
    Skills
    Education
    Experience

    Resume Text:
    {text}

    Return the result strictly in JSON format.
    """

    response = model.generate_content(prompt)

    return response.text


count = 0

for file in os.listdir(OCR_OUTPUT):

    if count >= 4:   # process only 4 resumes
        break

    with open(os.path.join(OCR_OUTPUT, file), "r") as f:
        text = f.read()

    result = extract_resume_information(text)

    json_file = file.replace(".txt", ".json")

    with open(os.path.join(JSON_OUTPUT, json_file), "w") as f:
        f.write(result)

    print(f"{json_file} created")

    count += 1

    # wait to avoid rate limit
    time.sleep(12)

print("Structured resume data extraction completed.")

resume_8.json created
resume_13.json created
resume_14.json created
resume_3.json created
Structured resume data extraction completed.


# Conclusion

In this project, we implemented an **Intelligent Document Processing (IDP) pipeline** to automatically extract structured information from resumes.

## Pipeline Summary

1. **Dataset Acquisition**
   - A resume dataset was downloaded from Kaggle using KaggleHub.

2. **Data Preparation**
   - A small subset of resumes was selected for efficient processing.
   - Resume PDFs were converted into image format.

3. **Image Preprocessing**
   - Images were converted to grayscale.
   - Noise reduction was applied using Gaussian blur.
   - Adaptive thresholding was used to improve text visibility.

4. **Optical Character Recognition (OCR)**
   - Tesseract OCR was used to extract textual content from the processed resume images.

5. **Information Extraction using LLM**
   - A Large Language Model (Gemini) analyzed the OCR text.
   - Key resume information such as **Name, Email, Phone, Skills, Education, and Experience** was extracted.

6. **Structured Output Generation**
   - The extracted information was converted into structured **JSON format**.

